<a href="https://colab.research.google.com/github/javirk/europa_surface/blob/revert_fixed/DEMO_apply_LineaMapper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Welcome to the demo of LineaMapper v1.1 and 2.0. In this jupyter notebook, you will be guided through ....

What do I want here?
- Apply LineaMapper with different configurations (and weights)
- use the apply_LineaMapper script
- download or save the georeferenced stitched output

In another notebook, I could guide through the window tiling algo.!?

Do I need a different environment? I can test in vgeosam312 to see if this works. --> no, does not work. I need geopoytorch. for gdal. I can do so by having a different environment.yml file. Or perhaps I can install gdal inside binder?
For inference, I do not need a GPU.

Also, the retrieval of length and width etc can be published later with the azimuth paper.

I can use apply_LineaMapper_all_RegionA.py as the basis

this script retrieves predictions on region A for the publication Haslebacher et al. (2024/2025)
for LineaMapper v1.0
   on 224 geosize
for LineaMapper v1.1
    on 224 geosize (tiles)
    on 112 geosize
for LineaMapper v2.0
   on 112 geosize

In [ ]:
IS_COLAB = False # execute this cell if you are NOT on Google colab, but on binder

In [1]:
IS_COLAB = True # execute this cell if you ARE on Google colab

In [2]:
import os
from pathlib import Path
from datetime import datetime
import time
from osgeo import ogr

In [3]:
# The below line clones the github repository to your local or remote machine
!git clone -b revert_fixed https://github.com/javirk/europa_surface.git

Cloning into 'europa_surface'...
remote: Enumerating objects: 2020, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 2020 (delta 40), reused 40 (delta 27), pack-reused 1940 (from 2)
Receiving objects: 100% (2020/2020), 3.93 MiB | 22.26 MiB/s, done.
Resolving deltas: 100% (1627/1627), done.


In [4]:
# we need to install the geojson module
!pip install geojson

We now clone the full github repository directly into Google Colab so that we can access every script. We change directory so that we are inside the cloned repository.

In [5]:
# we access the repository to import modules below
# also ONLY IF NOT ON BINDER:
if IS_COLAB == True:
    %cd europa_surface
    # and we need to install gdal in google Colab (following https://stackoverflow.com/questions/70275565/how-to-install-gdal-on-google-colab-fast)
    # this is for command line tools, which we'll use later
    !apt install gdal-bin


/content/europa_surface
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  python3-gdal python3-numpy
Suggested packages:
  libgdal-grass python-numpy-doc python3-pytest
The following NEW packages will be installed:
  gdal-bin python3-gdal python3-numpy
0 upgraded, 3 newly installed, 0 to remove and 30 not upgraded.
Need to get 5,055 kB of archives.
After this operation, 25.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-numpy amd64 1:1.21.5-1ubuntu22.04.1 [3,467 kB]
Get:2 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy/main amd64 python3-gdal amd64 3.6.4+dfsg-1~jammy0 [1,027 kB]
Get:3 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy/main amd64 gdal-bin amd64 3.6.4+dfsg-1~jammy0 [561 kB]
Fetched 5,055 kB in 4s (1,179 kB/s)
Selecting previously unselected package python3-numpy.
(Reading database ..

If you want to run LineaMapper on your own geotiff, simply put/upload your geotiff to the folder './demo/v1_1' (and remove everything for which you do not want predictions). Here, we run LineaMapper on the geotiff in './demo/v1_1' of region A from the publication.

In [6]:
# define the input path
# note: basepath is used to define the savepath
basepath = Path('.')

# we get this from the github repository
source_path = basepath / './demo/v1_1'
# because the next line catches every file that is ending in '.tif', you can also upload your own geotiff file here
tifpaths = sorted(source_path.glob('*.tif'))

dt_string = datetime.now().strftime("%Y_%m_%d")

Next, we download the weights for LineaMapper version 1.0, 1.1 and 2.0 and put them into a subdirectory './ckpts' (checkpoints). If for any reason, the direct download form Mendeley fails, go to the Mendeley and download the weights manually and put them into a sub-directory called 'ckpts' in the directory where this notebook is located.

In [15]:
import requests

# LineaMapper v1.0 weights: from Mendeley data repo
# url = "https://data.mendeley.com/public-files/datasets/nxnwj6hy4s/files/b248180d-aacb-43b4-a541-0b60d8076a3e/file_downloaded"
url = "https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/nxnwj6hy4s-1.zip"

# LM 1.0
# https://data.mendeley.com/api/datasets/rjhsjrnxgv/files/1815514f-a8b9-40cc-9822-1a3e41fc56d0/file_downloaded
# url = "https://data.mendeley.com/api/datasets/rjhsjrnxgv/draft/files/1815514f-a8b9-40cc-9822-1a3e41fc56d0"

# Download the file
response = requests.get(url)
response.raise_for_status()  # Ensure the request was successful

with open("LM10.zip", "wb") as file:
    file.write(response.content)

# Save the file locally, in a subfolder called 'ckpts' (for checkpoints)
%mkdir ckpts
with open("./ckpts/Weights_v1_0.pt", "wb") as file:
    file.write(response.content)

In [ ]:
# We define a straightforward function to execute LineaMapper_v1.py or ..._v2.py that we can call later for different models
# this function automatically loops through all found tiff files
# it stores the output as a geojson and a shapefile, along with a text file with the parameters we used
def forward_LM(modelname, version, subset, geosize):
    # NOTE: the savepath does not yet need to exist
    savepath = basepath / 'output/LineaMapper_output' / (dt_string + '_RegionA_' + subset)
    # start full time
    full_time_start = time.time()

    for tifffile in tifpaths:
        tf = tifffile.stem
        print(tf)
        command = f'python LineaMapper_{version}_to_img.py --modelname={modelname} --geofile=' + str(source_path.joinpath(tf + '.tif')) + ' --savedir=' + str(savepath) + f' --class_scores 0.5 0.5 0.5 0.5 --cut_size=3000 --multiplication_factor=15 --azimuth_diff_range=25 --del_pxs=100 --geosize={geosize}'
        print(command)
        # os.system(command)
        print(os.popen(command).read()) # for jupyter notebook, we need this line to execute the command

    # measure time and simply print
    timesum = time.time() - full_time_start
    print('this script took {:.2f} seconds to execute. Makes {:.2f} hours.'.format(timesum, timesum/3600))
    # simple test:
    # python LineaMapper_to_img.py --geofile=z:/Groups/PIG/Caroline/isis/data/galileo/usgs_photogrammetrically/Europa_Mosaics_Equirectangular/E6ESCRATER01_GalileoSSI_Equi-cog.tif --savedir=z:/Groups/PIG/Caroline/isis/data/galileo/usgs_photogrammetrically/LineaMapper_output/tests

    ######## convert to shapefiles
    # problem is that I do not have the exact filename, so I retrieve it simply afterwards
    # make shape_file folder
    os.makedirs(Path(savepath) / 'shape_files', exist_ok=True)

    geojfiles = sorted((Path(savepath) / 'json_files').glob('*.geojson'))
    for geojfile in geojfiles:
        # convert to shapefile as well
        command = 'ogr2ogr -nlt POLYGON -skipfailures {} {}'.format((savepath / 'shape_files').joinpath(geojfile.stem + '.shp'), (savepath / 'json_files').joinpath(geojfile.stem + '.geojson'))
        print(command)
        # os.system(command)
        print(os.popen(command).read()) # for jupyter notebook, we need this line to execute the command

    # check if it worked for all
    geoshpfiles = sorted((Path(savepath) / 'shape_files').glob('*.shp'))

    if len(geoshpfiles) == len(geojfiles):
        print('ALL GEOJSON FILES WERE CONVERTED TO SHAPEFILES.')
    else:
        raise Warning('some geojson files lead to errors, it seems.')

    return

Now, we are already prepared to actually run LineaMapper on our input geotiff image(s). This can take up to 1 hour (depending what computing power you have available).

In [ ]:
# for LineaMapper v1.0
#    on 224 geosize
subset = 'LM1.0_224' # identification string for savepath
geosize = 224
modelname = './ckpts/Mask_R-CNN_pub2_run23_end_model.pt'
version = 'v1' # 'v1' or 'v2'
# main
forward_LM(modelname, version, subset, geosize)



17ESREGMAP02_Bland2021_regionB
python LineaMapper_v1_to_img.py --modelname=./ckpts/Mask_R-CNN_pub2_run23_end_model.pt --geofile=demo/v1_1/17ESREGMAP02_Bland2021_regionB.tif --savedir=output/LineaMapper_output/2025_04_10_RegionA_LM1.0_224 --class_scores 0.5 0.5 0.5 0.5 --cut_size=3000 --multiplication_factor=15 --azimuth_diff_range=25 --del_pxs=100 --geosize=224

this script took 23.71 seconds to execute. Makes 0.01 hours.
ALL GEOJSON FILES WERE CONVERTED TO SHAPEFILES.


In [ ]:
# for LineaMapper v1.1
#     on 224 geosize (tiles)
subset = 'LM1.1_224' # identification string for savepath
geosize = 224
modelname = "./ckpts/Mask_R-CNN_v1_1_17ESREGMAP02_part01_run10_end_model.pt"
version = 'v1' # 'v1' or 'v2'
# main
forward_LM(modelname, version, subset, geosize)
#     on 112 geosize
geosize = 112
subset = 'LM1.1_112'
forward_LM(modelname, version, subset, geosize)

TypeError: unsupported operand type(s) for /: 'str' and 'str'

In [ ]:
# for LineaMapper v2.0
#     on 112 geosize
# note that sampath and sammodus are the default. "./ckpts/bbox_vit_b_final.pt", 'vit_b'
subset = 'LM2.0_112' # identification string for savepath
geosize = 112
modelname = "./ckpts/Mask_R-CNN_v1_1_17ESREGMAP02_part01_run10_end_model.pt"
version = 'v2' # 'v1' or 'v2'
# main
forward_LM(modelname, version, subset, geosize)

If the Mendeley data download fails, try this:

In [ ]:
import requests

# LineaMapper v1.0 weights: from Mendeley data repo for Haslebacher et al. (2024)
url = "https://data.mendeley.com/public-files/datasets/nxnwj6hy4s/files/b248180d-aacb-43b4-a541-0b60d8076a3e/file_downloaded"

# Download the file
response = requests.get(url)
response.raise_for_status()  # Ensure the request was successful

# Save the file locally, in a subfolder called 'ckpts' (for checkpoints)
%mkdir ckpts
with open("./ckpts/Weights_v1_0.pt", "wb") as file:
    file.write(response.content)

In [ ]:
# for LineaMapper v1.0
#    on 224 geosize
subset = 'LM1.0_224' # identification string for savepath
geosize = 224
modelname = './ckpts/Weights_v1_0.pt'
version = 'v1' # 'v1' or 'v2'
# main
forward_LM(modelname, version, subset, geosize)

17ESREGMAP02_Bland2021_regionB
python LineaMapper_v1_to_img.py --modelname=./ckpts/Weights_v1_0.pt --geofile=demo/v1_1/17ESREGMAP02_Bland2021_regionB.tif --savedir=output/LineaMapper_output/2025_04_10_RegionA_LM1.0_224 --class_scores 0.5 0.5 0.5 0.5 --cut_size=3000 --multiplication_factor=15 --azimuth_diff_range=25 --del_pxs=100 --geosize=224

this script took 8.11 seconds to execute. Makes 0.00 hours.
ALL GEOJSON FILES WERE CONVERTED TO SHAPEFILES.
